# PSR-4 End-to-End Scoring Notebook

Fetches OOS data directly from Snowflake, applies preprocessing, scores with the champion XGBoost model, and writes the enriched output back to Snowflake.


## 1. Imports & Configuration

In [1]:
import os
import json
import pickle
import warnings

import numpy as np
import pandas as pd
import shap
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

from io import BytesIO
from datetime import datetime
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')

/home/sagemaker-user/.conda/envs/sklearn_compat_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
# ==========================================
# CONFIGURATION
# ==========================================

# --- Snowflake ---
SNOWFLAKE_CONFIG = {
    "user":      'SVC_KNIME_DEV',
    "password":  '1bhYzI$Nq1evxo=',
    "account":   'ti97672.east-us-2.azure',
    "database":  'WORKSPACE_DS_OFFSHORE',
    "schema":    'MODEL',
    "warehouse": 'DEV_DW_DEVELOPER',
    "role":      'ZDI_DS_OFFSHORE_DEVELOPER',   # ← add this
}


INPUT_TABLE  = 'WORKSPACE_DS_OFFSHORE.MODEL.PSR4_PSR5_RF_TRAINING_AUG_2026'
OUTPUT_TABLE = 'PSR4_OUTPUT_DEV'

# --- Date filter (matches input_data.py) ---
OOS_START_DATE = '2026-01-01'   # OOS = DATE_END >= this
DATE_COLUMN    = 'DATE_END'

# --- Model artifact (local) ---



SCRIPT_DIR          = os.path.abspath('')
CUTOFFS_PATH        = os.path.join(SCRIPT_DIR, 'train_cutoffs1.json')
DECILE_CUTOFFS_PATH = os.path.join(SCRIPT_DIR, 'train_decile_cutoffs1.json')
ENCODER_PATH        = os.path.join(SCRIPT_DIR, 'train_encoder_providertype1.pkl')
MODEL_PATH = os.path.join(SCRIPT_DIR,'classification_psr4_project__XGBClassifier__20260813_065552.pkl')
# --- Column config ---
OOS_TARGET_COL     = 'OPT_IN_ACH'
OOS_TIME_COL       = 'DURATION'

TOP_N_DRIVERS   = 5
SHAP_THRESHOLD  = 0.005   # drivers below this |SHAP| don't count; replaced with None
CATEGORY_LABELS = ['Very Low', 'Low', 'Medium', 'High', 'Very High']

In [ ]:
# # ==========================================
# # CONFIGURATION
# # ==========================================

# # --- Snowflake ---
# SNOWFLAKE_CONFIG = {
#     "user":      'SVC_KNIME_DEV',
#     "password":  '1bhYzI$Nq1evxo=',
#     "account":   'ti97672.east-us-2.azure',
#     "database":  'WORKSPACE_DS_OFFSHORE',
#     "schema":    'MODEL',
#     "warehouse": 'DEV_DW_DEVELOPER'      # ← changed from DEV_DATA_ANALYST
# }

# INPUT_TABLE  = 'WORKSPACE_DS_OFFSHORE.MODEL.PSR4_PSR5_RF_TRAINING_AUG_2026'
# #OUTPUT_TABLE = 'PSR4_OUTPUT_DEV'

# # --- Date filter (matches input_data.py) ---
# OOS_START_DATE = '2026-01-01'   # OOS = DATE_END >= this
# OOS_END_DATE   = '2026-02-01'   # OOS: DATE_END <= this
# DATE_COLUMN    = 'DATE_END'

# # --- Model artifact (local) ---



# SCRIPT_DIR          = os.path.abspath('')
# CUTOFFS_PATH        = os.path.join(SCRIPT_DIR, 'train_cutoffs1.json')
# DECILE_CUTOFFS_PATH = os.path.join(SCRIPT_DIR, 'train_decile_cutoffs1.json')
# ENCODER_PATH        = os.path.join(SCRIPT_DIR, 'train_encoder_providertype1.pkl')
# MODEL_PATH = os.path.join(SCRIPT_DIR,'classification_psr4_project__XGBClassifier__20260813_065552.pkl')
# # --- Column config ---
# OOS_TARGET_COL     = 'OPT_IN_ACH'
# OOS_TIME_COL       = 'DURATION'

# TOP_N_DRIVERS   = 5
# SHAP_THRESHOLD  = 0.005   # drivers below this |SHAP| don't count; replaced with None
# CATEGORY_LABELS = ['Very Low', 'Low', 'Medium', 'High', 'Very High']

In [12]:
# ==========================================
# FEATURE DESCRIPTIONS (for human-readable drivers)
# ==========================================
FEATURE_DESCRIPTIONS = {
    'PROVIDERTYPE_MEDICAL':
        'Provider type is Medical',
    'PAYERID_TOTAL_COUNT':
        'Number of distinct payers associated with the account',
    'SUMCHECKPAYMENTCOUNT90':
        'Total number of check payments in the last 90 days',
    'CANCELLEDCHECKPAYMENTCOUNT':
        'Total cancelled check payment count (lifetime)',
    'PHONECOUNT90':
        'Number of phone interactions in the last 90 days',
    'TIME_SINCE_LAST_CANCELLED_CHECK_PAYMENT':
        'Days since the most recent cancelled check payment',
    'TOTAL_PREV_OPTOUTS_LAST_YEAR':
        'Number of prior ACH opt-outs in the last 12 months',
    'SUMACHPAYMENTCOUNT':
        'Total number of ACH payments (lifetime)',
    'CHECK_PAYMENT_AVG_30':
        'Average check payment amount in the last 30 days',
    'CHECK_PAYMENTCOUNT_MOMENTUM_30_90':
        'Ratio of daily check payment rate (last 30 days) to daily check payment rate (last 90 days)',
    'FIRSTACTION_BOOL':
        'Whether this is a brand-new account (created on snapshot start)',
    'ACCOUNT_AGE':
        'Age of the account in days since creation',
}


## 2. Helper Functions

In [5]:
def fetch_oos_from_snowflake(sf_config, input_table, date_col, oos_start_date):
    """Pulls OOS rows from Snowflake using DATE_END >= oos_start_date."""
    query = f"""
        SELECT *
        FROM {input_table}
        WHERE {date_col} >= '{oos_start_date}';
    """
    print(f"🔌 Connecting to Snowflake and pulling OOS rows...")
    print(f"   Query: {date_col} >= {oos_start_date}")
    conn = snowflake.connector.connect(**sf_config)
    try:
        df = pd.read_sql(query, conn)
        df.columns = [c.upper() for c in df.columns]
        print(f"   ✅ Retrieved {df.shape[0]:,} OOS rows.")
        return df
    finally:
        conn.close()

# def fetch_oos_from_snowflake(sf_config, input_table, date_col, oos_start_date, oos_end_date):
#     """Pulls OOS rows from Snowflake with DATE_END between start and end (both inclusive)."""
#     query = f"""
#         SELECT *
#         FROM {input_table}
#         WHERE {date_col} >= '{oos_start_date}'
#           AND {date_col} <= '{oos_end_date}';
#     """
#     print(f"🔌 Connecting to Snowflake and pulling OOS rows...")
#     print(f"   Query: {date_col} between {oos_start_date} and {oos_end_date} (inclusive)")
#     conn = snowflake.connector.connect(**sf_config)
#     try:
#         df = pd.read_sql(query, conn)
#         df.columns = [c.upper() for c in df.columns]
#         print(f"   ✅ Retrieved {df.shape[0]:,} OOS rows.")
#         return df
#     finally:
#         conn.close()


def categorize_probability(prob, cutoffs):
    """Assign a 5-tier category based on train-derived cutoffs."""
    if prob >= cutoffs['Very_High']:
        return 'Very High'
    elif prob >= cutoffs['High']:
        return 'High'
    elif prob >= cutoffs['Medium']:
        return 'Medium'
    elif prob >= cutoffs['Low']:
        return 'Low'
    else:
        return 'Very Low'

def assign_decile(prob, decile_cutoffs):
    """Assign a decile bucket (1 = highest, 10 = lowest) using train-derived decile thresholds."""
    if   prob >= decile_cutoffs['d10_min']: return 1
    elif prob >= decile_cutoffs['d9_min']:  return 2
    elif prob >= decile_cutoffs['d8_min']:  return 3
    elif prob >= decile_cutoffs['d7_min']:  return 4
    elif prob >= decile_cutoffs['d6_min']:  return 5
    elif prob >= decile_cutoffs['d5_min']:  return 6
    elif prob >= decile_cutoffs['d4_min']:  return 7
    elif prob >= decile_cutoffs['d3_min']:  return 8
    elif prob >= decile_cutoffs['d2_min']:  return 9
    else:                                    return 10

def extract_top_drivers(shap_values, feature_values, feature_names, top_n=5, threshold=0.005):
    """
    For each row, identify the top-N features by |SHAP value|, filtering out
    any driver whose SHAP magnitude is below the threshold (replaced with None).
    """
    n_rows   = shap_values.shape[0]
    abs_shap = np.abs(shap_values)

    feature_names_arr  = np.array(feature_names)
    feature_values_arr = feature_values.values

    output = {}
    for rank in range(top_n):
        output[f'OPTINDRIVER_{rank+1}']        = [None] * n_rows
        output[f'OPTINDRIVER_{rank+1}_VALUE']  = [None] * n_rows

    for i in range(n_rows):
        row_abs   = abs_shap[i]
        mask      = row_abs >= threshold
        if not mask.any():
            continue
        valid_idx = np.where(mask)[0]
        # Sort valid indices by |SHAP| descending
        sorted_idx = valid_idx[np.argsort(row_abs[valid_idx])[::-1]]
        top_idx    = sorted_idx[:top_n]
        for rank, feat_idx in enumerate(top_idx):
            raw_name = feature_names_arr[feat_idx]
            output[f'OPTINDRIVER_{rank+1}'][i]        = FEATURE_DESCRIPTIONS.get(raw_name, raw_name)
            output[f'OPTINDRIVER_{rank+1}_VALUE'][i]  = feature_values_arr[i, feat_idx]

    return pd.DataFrame(output)

## 3. STAGE 1 — Preprocessing

In [6]:
def run_base_preprocessing(df, time_col):
    """Apply base filters, null-fills, and compute the 4 derived features."""
    # --- Base filters ---
    df = df.drop_duplicates().reset_index(drop=True)

    # PAYERSPONSORED can come back as bool OR string depending on the source
    # (Snowflake direct read → bool; CSV round-trip → 'True'/'False' strings).
    # Compare as lowercase string to be robust to both.
    df = df[df['PAYERSPONSORED'].astype(str).str.lower() != 'true']

    fill_zero_cols = [
        'CANCELLEDCHECKPAYMENTCOUNT', 'CANCELLEDCHECKPAYMENTCOUNT30', 'CANCELLEDCHECKPAYMENTCOUNT60', 'CANCELLEDCHECKPAYMENTCOUNT90',
        'CANCELLEDCHECKPAYMENTAMOUNT', 'CANCELLEDCHECKPAYMENTAMOUNT30', 'CANCELLEDCHECKPAYMENTAMOUNT60', 'CANCELLEDCHECKPAYMENTAMOUNT90',
        'SUMCHECKPAYMENTAMOUNT', 'SUMCHECKPAYMENTCOUNT', 'SUMCHECKPAYMENTAMOUNT30', 'SUMCHECKPAYMENTCOUNT30',
        'SUMCHECKPAYMENTAMOUNT60', 'SUMCHECKPAYMENTCOUNT60', 'SUMCHECKPAYMENTAMOUNT90', 'SUMCHECKPAYMENTCOUNT90',
        'SUMACHPAYMENTAMOUNT', 'SUMACHPAYMENTCOUNT', 'SUMVCCPAYMENTAMOUNT', 'SUMVCCPAYMENTCOUNT',
        'SUMPAYERSPONSOREDPAYMENTAMOUNT', 'SUMPAYERSPONSOREDPAYMENTCOUNT', 'SUMTOTALPAYMENTAMOUNT', 'SUMTOTALPAYMENTCOUNT',
        'PREVIOUSOPTINCOUNT', 'PREVIOUSACHOPTINCOUNT', 'PREVIOUSVCCOPTINCOUNT',
        'PHONECOUNT', 'PHONECOUNT30', 'PHONECOUNT60', 'PHONECOUNT90',

    ]
    for col in fill_zero_cols:
        if col in df.columns:
            df[col].fillna(0, inplace=True)

    if 'TIME_SINCE_LAST_CANCELLED_CHECK_PAYMENT' in df.columns:
        df['TIME_SINCE_LAST_CANCELLED_CHECK_PAYMENT'].fillna(9999, inplace=True)
    if 'TIME_SINCE_LAST_CHECK_PAYMENT' in df.columns:
        df['TIME_SINCE_LAST_CHECK_PAYMENT'].fillna(9999, inplace=True)

    df = df[df['SUMCHECKPAYMENTCOUNT'] > 0]
    if time_col in df.columns:
        df = df[df[time_col] >= 90]

    # --- Derived feature #1: CHECK_PAYMENT_AVG_30 ---
    df['CHECK_PAYMENT_AVG_30'] = df['SUMCHECKPAYMENTAMOUNT30'] / df['SUMCHECKPAYMENTCOUNT30'].replace(0, 1)
    df['CHECK_PAYMENT_AVG_30'].fillna(0, inplace=True)

    # --- Derived feature #2: CHECK_PAYMENTCOUNT_MOMENTUM_30_90 ---
    _rate_30 = df['SUMCHECKPAYMENTCOUNT30'] / 30
    _rate_90 = df['SUMCHECKPAYMENTCOUNT90'] / 90
    df['CHECK_PAYMENTCOUNT_MOMENTUM_30_90'] = _rate_30 / _rate_90.replace(0, 0.001)
    df['CHECK_PAYMENTCOUNT_MOMENTUM_30_90'].fillna(-1, inplace=True)

    # --- Derived feature #3: FIRSTACTION_BOOL ---
    if 'PROVIDERCREATEDON' in df.columns and 'DATE_START' in df.columns:
        df['FIRSTACTION_BOOL'] = (df['PROVIDERCREATEDON'] == df['DATE_START']).astype(int)

    # --- Derived feature #4: ACCOUNT_AGE (days between DATE_END and PROVIDERCREATEDON) ---
    if 'SNAPSHOT_END_DATE' in df.columns and 'PROVIDERCREATEDON' in df.columns:
        df['ACCOUNT_AGE'] = (
            pd.to_datetime(df['SNAPSHOT_END_DATE']) - pd.to_datetime(df['PROVIDERCREATEDON'])
        ).dt.days
    elif 'DATE_END' in df.columns and 'PROVIDERCREATEDON' in df.columns:
        df['ACCOUNT_AGE'] = (
            pd.to_datetime(df['DATE_END']) - pd.to_datetime(df['PROVIDERCREATEDON'])
        ).dt.days

    df.reset_index(drop=True, inplace=True)

    # --- Categorical dtype (only PROVIDERTYPE; it's the one being one-hot encoded) ---
    if 'PROVIDERTYPE' in df.columns:
        df['PROVIDERTYPE'] = df['PROVIDERTYPE'].astype('category')

    return df


def execute_encoding(df_raw, encoder_path):
    """
    Apply the train-fit OneHotEncoder to PROVIDERTYPE; everything else passes through as-is.

    Loads the encoder from a pickle file (fit on training data), so OOS
    encoding produces the exact same columns / ordering as train. This is the
    ML best practice — never fit a fresh encoder on inference data.
    """
    df_in = df_raw.reset_index(drop=True).copy()

    exclude_cols     = ['TIN', OOS_TARGET_COL]
    cat_cols         = ['PROVIDERTYPE'] if 'PROVIDERTYPE' in df_in.columns else []
    passthrough_cols = [c for c in df_in.columns if c not in exclude_cols and c not in cat_cols]

    print(f"   Encoding: {cat_cols}")
    print(f"   Passthrough ({len(passthrough_cols)} cols): kept as-is")

    if cat_cols:
        print(f"   Loading train-fit encoder from: {encoder_path}")
        with open(encoder_path, 'rb') as f:
            encoder = pickle.load(f)
        print(f"   Encoder learned categories: {list(encoder.categories_[0])}")

        encoded_cats = encoder.transform(df_in[cat_cols])
        df_cat = pd.DataFrame(encoded_cats, columns=encoder.get_feature_names_out(cat_cols))
    else:
        df_cat = pd.DataFrame()

    df_passthrough = df_in[passthrough_cols].reset_index(drop=True)
    df_final = pd.concat([df_in[['TIN', OOS_TARGET_COL]].reset_index(drop=True),
                          df_passthrough,
                          df_cat], axis=1)
    return df_final


In [7]:
# ---- Run Stage 1 ----
print("=" * 70)
print("[STAGE 1] Preprocessing")
print("=" * 70)

df_raw = fetch_oos_from_snowflake(SNOWFLAKE_CONFIG, INPUT_TABLE, DATE_COLUMN, OOS_START_DATE)
# df_raw = fetch_oos_from_snowflake(SNOWFLAKE_CONFIG, INPUT_TABLE, DATE_COLUMN, OOS_START_DATE, OOS_END_DATE)

print(f"Raw OOS shape: {df_raw.shape}")

print("\nRunning base preprocessing...")
df_processed = run_base_preprocessing(df_raw, OOS_TIME_COL)
print(f"After preprocessing shape: {df_processed.shape}")

print("\nRunning encoding step (using train-fit encoder)...")
df_encoded = execute_encoding(df_processed, ENCODER_PATH)
print(f"After encoding shape: {df_encoded.shape}")

[STAGE 1] Preprocessing
🔌 Connecting to Snowflake and pulling OOS rows...
   Query: DATE_END >= 2026-01-01


   ✅ Retrieved 3,894,827 OOS rows.
Raw OOS shape: (3894827, 69)

Running base preprocessing...


After preprocessing shape: (3222906, 73)

Running encoding step (using train-fit encoder)...


   Encoding: ['PROVIDERTYPE']
   Passthrough (70 cols): kept as-is
   Loading train-fit encoder from: /home/sagemaker-user/PSR4(8th June)/train_encoder_providertype1.pkl
   Encoder learned categories: ['DENTAL', 'MEDICAL']


After encoding shape: (3222906, 74)


## 4. STAGE 2 — Scoring

In [9]:
# ---- Load model, cutoffs, decile cutoffs ----
print("=" * 70)
print("[STAGE 2] Scoring")
print("=" * 70)

print(f"\nLoading champion model from local: {MODEL_PATH}")
with open(MODEL_PATH, 'rb') as f:
    loaded_obj = pickle.load(f)

if isinstance(loaded_obj, dict) and 'model' in loaded_obj:
    model = loaded_obj['model']
    model_features = loaded_obj.get('features')
else:
    model = loaded_obj
    model_features = getattr(model, 'feature_names_in_', None)

if model_features is None:
    raise ValueError("Could not extract feature list from the model pickle.")
model_features = list(model_features)
print(f"Model expects {len(model_features)} features.")

print(f"\nLoading category cutoffs: {CUTOFFS_PATH}")
with open(CUTOFFS_PATH, 'r') as f:
    cutoffs_json = json.load(f)
cutoffs = cutoffs_json.get('cutoffs', cutoffs_json)
print(f"  Very_High={cutoffs['Very_High']:.4f} | High={cutoffs['High']:.4f} | "
      f"Medium={cutoffs['Medium']:.4f} | Low={cutoffs['Low']:.4f}")

print(f"\nLoading decile cutoffs: {DECILE_CUTOFFS_PATH}")
with open(DECILE_CUTOFFS_PATH, 'r') as f:
    decile_json = json.load(f)
decile_cutoffs = decile_json.get('cutoffs', decile_json)
print(f"  Decile 1 starts at: {decile_cutoffs['d10_min']:.4f}")
print(f"  Decile 9 threshold: {decile_cutoffs['d2_min']:.4f}")

[STAGE 2] Scoring

Loading champion model from local: /home/sagemaker-user/PSR4(8th June)/classification_psr4_project__XGBClassifier__20260813_065552.pkl
Model expects 12 features.

Loading category cutoffs: /home/sagemaker-user/PSR4(8th June)/train_cutoffs1.json
  Very_High=0.9084 | High=0.8075 | Medium=0.6120 | Low=0.2823

Loading decile cutoffs: /home/sagemaker-user/PSR4(8th June)/train_decile_cutoffs1.json
  Decile 1 starts at: 0.8075
  Decile 9 threshold: 0.0393


In [10]:
# ---- Compatibility check + Score ----
missing = [f for f in model_features if f not in df_encoded.columns]
if missing:
    raise ValueError(f"Preprocessed output is missing {len(missing)} model feature(s): {missing}")

X_oos = df_encoded[model_features].copy()
print(f"\nFeature matrix shape: {X_oos.shape}")

print("Scoring with model.predict_proba()...")
probabilities = model.predict_proba(X_oos)[:, 1]
print(f"Scored {len(probabilities):,} rows. Prob range: [{probabilities.min():.4f}, {probabilities.max():.4f}]")

# Categorize into 5 tiers
categories = np.array([categorize_probability(p, cutoffs) for p in probabilities])

# Assign decile (1 = top 10%, 10 = bottom 10%)
deciles = np.array([assign_decile(p, decile_cutoffs) for p in probabilities])


Feature matrix shape: (3222906, 12)
Scoring with model.predict_proba()...


Scored 3,222,906 rows. Prob range: [0.0004, 0.9989]


### Decile Table (10 bins, train-derived cutoffs)

In [11]:
# Build OOS decile analysis table
y_oos = df_encoded[OOS_TARGET_COL].astype(int).values
total_positives = int(y_oos.sum())
total_count     = len(y_oos)
overall_rate    = total_positives / total_count if total_count > 0 else 0

decile_rows = []
for d in range(1, 11):
    mask   = deciles == d
    cnt    = int(mask.sum())
    pos    = int(y_oos[mask].sum())
    rate   = (pos / cnt * 100) if cnt > 0 else 0
    overpct= (pos / total_positives * 100) if total_positives > 0 else 0
    lift   = (rate / 100) / overall_rate if overall_rate > 0 else 0
    prob_mask = probabilities[mask]
    decile_rows.append({
        'Decile':          d,
        'Counts':          cnt,
        'Positive Target': pos,
        'Target Rate %':   round(rate, 4),
        'Overall Target Captured %': round(overpct, 4),
        'Lift':            round(lift, 4),
        'Min Prob':        round(float(prob_mask.min()), 6) if cnt > 0 else None,
        'Max Prob':        round(float(prob_mask.max()), 6) if cnt > 0 else None,
    })

decile_df = pd.DataFrame(decile_rows)
print(f"\nOOS Decile Analysis (using train-derived cutoffs)")
print(f"Total rows: {total_count:,} | Total positives: {total_positives:,} | Baseline rate: {overall_rate*100:.4f}%")
decile_df


OOS Decile Analysis (using train-derived cutoffs)
Total rows: 3,222,906 | Total positives: 3,503 | Baseline rate: 0.1087%


,Decile,Counts,Positive Target,Target Rate %,Overall Target Captured %,Lift,Min Prob,Max Prob
0,1,310486,1974,0.6358,56.3517,5.8494,0.807497,0.998894
1,2,320096,729,0.2277,20.8107,2.0953,0.701365,0.807496
2,3,321981,403,0.1252,11.5044,1.1515,0.566473,0.701365
3,4,342523,227,0.0663,6.4802,0.6097,0.375652,0.566472
4,5,314522,104,0.0331,2.9689,0.3042,0.188012,0.375651
5,6,283216,30,0.0106,0.8564,0.0975,0.119935,0.188011
6,7,266065,9,0.0034,0.2569,0.0311,0.078942,0.119928
7,8,320238,16,0.0050,0.4568,0.0460,0.066788,0.078939
8,9,310140,5,0.0016,0.1427,0.0148,0.039308,0.066785
9,10,433639,6,0.0014,0.1713,0.0127,0.000353,0.039304


In [13]:
# ---- SHAP drivers with 0.005 threshold ----
print("Computing SHAP values (TreeExplainer)...")
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_oos)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

print(f"SHAP matrix shape: {shap_values.shape}")
print(f"Extracting top-{TOP_N_DRIVERS} drivers (threshold |SHAP| >= {SHAP_THRESHOLD})...")
drivers_df = extract_top_drivers(shap_values, X_oos, model_features,
                                  top_n=TOP_N_DRIVERS, threshold=SHAP_THRESHOLD)
print(f"Drivers dataframe shape: {drivers_df.shape}")

Computing SHAP values (TreeExplainer)...


SHAP matrix shape: (3222906, 12)
Extracting top-5 drivers (threshold |SHAP| >= 0.005)...


Drivers dataframe shape: (3222906, 10)


In [14]:
# ---- Assemble final output ----
df_encoded = df_encoded.reset_index(drop=True)
drivers_df = drivers_df.reset_index(drop=True)

df_out = df_encoded.copy()
df_out['OPT_IN_PROBABILITY']            = probabilities
df_out['OPT_IN_CATEGORY']               = categories
# df_out['OPT_IN_DECILE']                 = deciles
df_out = pd.concat([df_out, drivers_df], axis=1)
df_out['CAMPAIGN_LEADS_RECOMMENDATION'] = (df_out['OPT_IN_CATEGORY'] == 'Very High').astype(int)

# Sort: category (Very High → Very Low), then probability desc
category_order = pd.CategoricalDtype(
    categories=['Very High', 'High', 'Medium', 'Low', 'Very Low'],
    ordered=True
)
df_out['OPT_IN_CATEGORY'] = df_out['OPT_IN_CATEGORY'].astype(category_order)
df_out = df_out.sort_values(
    by=['OPT_IN_CATEGORY', 'OPT_IN_PROBABILITY'],
    ascending=[True, False]
).reset_index(drop=True)

df_out['OPT_IN_ORDER_WITHIN_CATEGORY'] = df_out.groupby('OPT_IN_CATEGORY').cumcount() + 1
df_out['CALL_TARGET_ORDER']            = np.arange(1, len(df_out) + 1)
df_out['OPT_IN_CATEGORY']              = df_out['OPT_IN_CATEGORY'].astype(str)

# Reorder new columns to the end
driver_cols = [c for c in df_out.columns if c.startswith('top_driver_')]
new_cols_ordered = (
    ['OPT_IN_PROBABILITY', 'OPT_IN_CATEGORY',
     'OPT_IN_ORDER_WITHIN_CATEGORY', 'CALL_TARGET_ORDER']
    + driver_cols
    + ['CAMPAIGN_LEADS_RECOMMENDATION']
)
other_cols = [c for c in df_out.columns if c not in new_cols_ordered]
df_out = df_out[other_cols + new_cols_ordered]

print(f"Final output shape: {df_out.shape}")
print(f"Very High rows (CAMPAIGN_LEADS = 1): {(df_out['CAMPAIGN_LEADS_RECOMMENDATION'] == 1).sum():,}")

# Category distribution
print("\nCategory distribution:")
dist = df_out['OPT_IN_CATEGORY'].value_counts(normalize=True).reindex(CATEGORY_LABELS[::-1]).fillna(0)
for cat, pct in dist.items():
    print(f"  {cat:<12} {pct*100:5.2f}%")

Final output shape: (3222906, 89)
Very High rows (CAMPAIGN_LEADS = 1): 58,382

Category distribution:


  Very High     1.81%
  High          7.82%
  Medium       16.95%
  Low          17.94%
  Very Low     55.48%



## 5. Write Output To Snowflake

Table: `WORKSPACE_DS_OFFSHORE.MODEL.PSR4_OUTPUT_DEV`



In [20]:
# ==========================================
# Push to Snowflake via write_pandas
# ==========================================

# Prep: sanitize column names + cast object cols to string
df_write = df_out.copy()

for col in df_write.columns:
    if df_write[col].dtype == object:
        df_write[col] = df_write[col].astype(str)

df_write.columns = [str(c).upper() for c in df_write.columns]

print(f"Writing {df_write.shape[0]:,} rows × {df_write.shape[1]} cols to {OUTPUT_TABLE}...")

conn = snowflake.connector.connect(**SNOWFLAKE_CONFIG)

# Show what role/warehouse we're on
cur = conn.cursor()
cur.execute("SELECT CURRENT_ROLE(), CURRENT_WAREHOUSE()")
role_wh = cur.fetchone()
print(f"Connected as role={role_wh[0]}, warehouse={role_wh[1]}")
cur.close()

try:
    success, nchunks, nrows, _ = write_pandas(
        conn, df_write,
        OUTPUT_TABLE,
        auto_create_table=True,
        overwrite=True
    )
    print(f"✅ Snowflake write complete: success={success} | rows={nrows:,} | chunks={nchunks}")
finally:
    conn.close()


Writing 3,222,906 rows × 89 cols to PSR4_OUTPUT_DEV...


Connected as role=ZDI_DS_OFFSHORE_DEVELOPER, warehouse=DEV_DW_DEVELOPER


✅ Snowflake write complete: success=True | rows=3,222,906 | chunks=1
